In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

import findspark

findspark.init()

# 1. Loading the Dataset into Apache Spark

In this section, Apache Spark is initialized and the cleaned Blood Cold Chain Monitoring dataset is loaded into a Spark DataFrame. The dataset prepared in Notebook 1 is used for all the distributed processing tasks performed in this notebook.

Spark DataFrames provide a distributed and fault-tolerant way of processing large datasets while supporting efficient analytical operations.

In [3]:
spark = (
    SparkSession.builder
    .appName("Blood Cold Chain Monitoring")
    .master("local[*]")
    .getOrCreate()
)

spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/08/01 22:03:31 WARN Utils: Your hostname, Khushis-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 192.168.0.101 instead (on interface en0)
26/08/01 22:03:31 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/Users/rohit/Documents/BigData_BloodColdChainMonitoring/.venv/lib/python3.12/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/08/01 22:03:32 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


# 2. Basic Spark Data Exploration

After loading the dataset, an initial exploration is performed using Spark DataFrame operations. This helps in understanding the dataset structure, data types, number of records, and key attributes before performing distributed processing.

Basic transformations and actions such as selecting columns, filtering records, grouping data, and sorting results are also demonstrated using healthcare telemetry data.

In [4]:
spark_df = spark.read.csv(
    "datasets/blood_cold_chain_cleaned.csv",
    header=True,
    inferSchema=True
)

spark_df.show(5)

+--------+-------------------+----------+----------+------------+-----------------+-----------------+------------------+------------------+------------------+------------------+-----------------+------------------+-------------------+-------------------+------------------+------------------+------------------+
|  bag_id|          timestamp|     route|blood_type|product_type|        temp_mean|         temp_min|          temp_max|          temp_std| frac_temp_above_6| frac_temp_above_8|         hum_mean|           hum_std|         door_count|     light_mean_abs|         accel_rms|   handling_stress|      health_index|
+--------+-------------------+----------+----------+------------+-----------------+-----------------+------------------+------------------+------------------+------------------+-----------------+------------------+-------------------+-------------------+------------------+------------------+------------------+
|BAG_0001|2024-01-11 00:00:00|Hospital_3|        B-|         RBC

In [5]:
spark_df.printSchema()

root
 |-- bag_id: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- route: string (nullable = true)
 |-- blood_type: string (nullable = true)
 |-- product_type: string (nullable = true)
 |-- temp_mean: double (nullable = true)
 |-- temp_min: double (nullable = true)
 |-- temp_max: double (nullable = true)
 |-- temp_std: double (nullable = true)
 |-- frac_temp_above_6: double (nullable = true)
 |-- frac_temp_above_8: double (nullable = true)
 |-- hum_mean: double (nullable = true)
 |-- hum_std: double (nullable = true)
 |-- door_count: double (nullable = true)
 |-- light_mean_abs: double (nullable = true)
 |-- accel_rms: double (nullable = true)
 |-- handling_stress: double (nullable = true)
 |-- health_index: double (nullable = true)



In [6]:
print(
    f"Rows: {spark_df.count()}"
)

print(
    f"Columns: {len(spark_df.columns)}"
)

Rows: 288000
Columns: 18


In [7]:
spark_df.show(10, truncate=False)

+--------+-------------------+----------+----------+------------+-----------------+------------------+------------------+------------------+------------------+------------------+-----------------+------------------+-------------------+-------------------+------------------+------------------+------------------+
|bag_id  |timestamp          |route     |blood_type|product_type|temp_mean        |temp_min          |temp_max          |temp_std          |frac_temp_above_6 |frac_temp_above_8 |hum_mean         |hum_std           |door_count         |light_mean_abs     |accel_rms         |handling_stress   |health_index      |
+--------+-------------------+----------+----------+------------+-----------------+------------------+------------------+------------------+------------------+------------------+-----------------+------------------+-------------------+-------------------+------------------+------------------+------------------+
|BAG_0001|2024-01-11 00:00:00|Hospital_3|B-        |RBC      

In [8]:
spark_df.describe().show()

26/08/01 22:06:36 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------+--------+----------+----------+------------+------------------+------------------+------------------+-------------------+--------------------+-------------------+------------------+------------------+-------------------+-------------------+--------------------+-------------------+-------------------+
|summary|  bag_id|     route|blood_type|product_type|         temp_mean|          temp_min|          temp_max|           temp_std|   frac_temp_above_6|  frac_temp_above_8|          hum_mean|           hum_std|         door_count|     light_mean_abs|           accel_rms|    handling_stress|       health_index|
+-------+--------+----------+----------+------------+------------------+------------------+------------------+-------------------+--------------------+-------------------+------------------+------------------+-------------------+-------------------+--------------------+-------------------+-------------------+
|  count|  288000|    288000|    288000|      288000|            28

In [9]:
spark_df.select(
    "route",
    "blood_type",
    "product_type"
).show(10)

+----------+----------+------------+
|     route|blood_type|product_type|
+----------+----------+------------+
|Hospital_3|        B-|         RBC|
|Hospital_3|        B-|         RBC|
|Hospital_3|        B-|         RBC|
|Hospital_3|        B-|         RBC|
|Hospital_3|        B-|         RBC|
|Hospital_3|        B-|         RBC|
|Hospital_3|        B-|         RBC|
|Hospital_3|        B-|         RBC|
|Hospital_3|        B-|         RBC|
|Hospital_3|        B-|         RBC|
+----------+----------+------------+
only showing top 10 rows


In [10]:
spark_df.select(
    "bag_id",
    "route",
    "temp_mean",
    "health_index"
).show(10)

+--------+----------+-----------------+------------------+
|  bag_id|     route|        temp_mean|      health_index|
+--------+----------+-----------------+------------------+
|BAG_0001|Hospital_3|              3.5|0.9941757563161104|
|BAG_0001|Hospital_3|              3.5|0.9874076533236108|
|BAG_0001|Hospital_3|4.339103727877417| 0.980271022375168|
|BAG_0001|Hospital_3|              3.5|0.9731695981310298|
|BAG_0001|Hospital_3|              3.5|0.9665432160148096|
|BAG_0001|Hospital_3|              3.5|0.9592298910746802|
|BAG_0001|Hospital_3|              3.5| 0.936047805934561|
|BAG_0001|Hospital_3|3.742637655914638|0.9298687609533496|
|BAG_0001|Hospital_3|              3.5| 0.924114144397302|
|BAG_0001|Hospital_3|              3.5|0.9018147848072104|
+--------+----------+-----------------+------------------+
only showing top 10 rows


In [11]:
high_temp_df = spark_df.filter(
    col("temp_mean") > 6
)

high_temp_df.show()

+--------+-------------------+----------+----------+------------+------------------+------------------+-----------------+------------------+-------------------+-------------------+------------------+------------------+-------------------+-------------------+------------------+------------------+--------------------+
|  bag_id|          timestamp|     route|blood_type|product_type|         temp_mean|          temp_min|         temp_max|          temp_std|  frac_temp_above_6|  frac_temp_above_8|          hum_mean|           hum_std|         door_count|     light_mean_abs|         accel_rms|   handling_stress|        health_index|
+--------+-------------------+----------+----------+------------+------------------+------------------+-----------------+------------------+-------------------+-------------------+------------------+------------------+-------------------+-------------------+------------------+------------------+--------------------+
|BAG_0004|2024-01-05 17:00:00|Hospital_2|     

In [12]:
spark_df.groupBy("route").agg(
    avg("temp_mean").alias("Average Temperature")
).show()

+----------+-------------------+
|     route|Average Temperature|
+----------+-------------------+
|Hospital_4| 4.2538828063075345|
|Hospital_1|  4.241393266763376|
|Hospital_3|   4.19832277153639|
|Hospital_2|  4.164778584459531|
+----------+-------------------+



In [13]:
spark_df.orderBy(
    col("handling_stress").desc()
).show(10)

+--------+-------------------+----------+----------+------------+------------------+------------------+------------------+------------------+-------------------+------------------+-----------------+------------------+------------------+-------------------+------------------+------------------+--------------------+
|  bag_id|          timestamp|     route|blood_type|product_type|         temp_mean|          temp_min|          temp_max|          temp_std|  frac_temp_above_6| frac_temp_above_8|         hum_mean|           hum_std|        door_count|     light_mean_abs|         accel_rms|   handling_stress|        health_index|
+--------+-------------------+----------+----------+------------+------------------+------------------+------------------+------------------+-------------------+------------------+-----------------+------------------+------------------+-------------------+------------------+------------------+--------------------+
|BAG_0273|2024-01-27 10:00:00|Hospital_2|        A-|

## 3. Spark SQL Operations

Apache Spark provides a SQL interface that allows structured queries to be executed on distributed datasets. In this section, the cleaned healthcare IoT dataset is registered as a temporary SQL view, and SQL queries are used to analyze transportation routes, blood types, and handling stress during blood transportation.

In [14]:
# Register DataFrame as a temporary SQL view
spark_df.createOrReplaceTempView("blood_monitoring")

In [15]:
spark.sql("""
SELECT
    route,
    AVG(temp_mean) AS average_temperature
FROM blood_monitoring
GROUP BY route
ORDER BY average_temperature DESC
""").show()

+----------+-------------------+
|     route|average_temperature|
+----------+-------------------+
|Hospital_4| 4.2538828063075345|
|Hospital_1|  4.241393266763376|
|Hospital_3|   4.19832277153639|
|Hospital_2|  4.164778584459531|
+----------+-------------------+



In [16]:
spark.sql("""
SELECT
    blood_type,
    COUNT(*) AS total_bags
FROM blood_monitoring
GROUP BY blood_type
ORDER BY total_bags DESC
""").show()

+----------+----------+
|blood_type|total_bags|
+----------+----------+
|        O+|     44160|
|        A-|     41280|
|        A+|     39360|
|       AB-|     37440|
|       AB+|     36480|
|        B+|     32640|
|        B-|     30720|
|        O-|     25920|
+----------+----------+



In [17]:
spark.sql("""
SELECT
    bag_id,
    route,
    handling_stress
FROM blood_monitoring
ORDER BY handling_stress DESC
LIMIT 10
""").show()

+--------+----------+------------------+
|  bag_id|     route|   handling_stress|
+--------+----------+------------------+
|BAG_0273|Hospital_2|2.6448267082470744|
|BAG_0195|Hospital_2|2.6400015178000333|
|BAG_0231|Hospital_1|2.4992905597246806|
|BAG_0030|Hospital_4|2.4732927866205108|
|BAG_0171|Hospital_2| 2.472723569806285|
|BAG_0135|Hospital_1|2.4457264383571085|
|BAG_0035|Hospital_4|2.4315158620907504|
|BAG_0021|Hospital_1|2.4287266158999947|
|BAG_0070|Hospital_3|2.4151159352353324|
|BAG_0090|Hospital_2|2.3726791131903338|
+--------+----------+------------------+



In [19]:
# Register the DataFrame as a temporary SQL view
spark_df.createOrReplaceTempView("blood_monitoring")

In [20]:
spark.sql("""
SELECT
    route,
    ROUND(AVG(temp_mean), 2) AS average_temperature
FROM blood_monitoring
GROUP BY route
ORDER BY average_temperature DESC
""").show()

+----------+-------------------+
|     route|average_temperature|
+----------+-------------------+
|Hospital_4|               4.25|
|Hospital_1|               4.24|
|Hospital_3|                4.2|
|Hospital_2|               4.16|
+----------+-------------------+



In [21]:
spark.sql("""
SELECT
    blood_type,
    COUNT(*) AS total_bags
FROM blood_monitoring
GROUP BY blood_type
ORDER BY total_bags DESC
""").show()

+----------+----------+
|blood_type|total_bags|
+----------+----------+
|        O+|     44160|
|        A-|     41280|
|        A+|     39360|
|       AB-|     37440|
|       AB+|     36480|
|        B+|     32640|
|        B-|     30720|
|        O-|     25920|
+----------+----------+



In [22]:
spark.sql("""
SELECT
    bag_id,
    route,
    handling_stress
FROM blood_monitoring
ORDER BY handling_stress DESC
LIMIT 10
""").show()

+--------+----------+------------------+
|  bag_id|     route|   handling_stress|
+--------+----------+------------------+
|BAG_0273|Hospital_2|2.6448267082470744|
|BAG_0195|Hospital_2|2.6400015178000333|
|BAG_0231|Hospital_1|2.4992905597246806|
|BAG_0030|Hospital_4|2.4732927866205108|
|BAG_0171|Hospital_2| 2.472723569806285|
|BAG_0135|Hospital_1|2.4457264383571085|
|BAG_0035|Hospital_4|2.4315158620907504|
|BAG_0021|Hospital_1|2.4287266158999947|
|BAG_0070|Hospital_3|2.4151159352353324|
|BAG_0090|Hospital_2|2.3726791131903338|
+--------+----------+------------------+



In [23]:
spark.sql("""
SELECT
    route,
    ROUND(AVG(health_index), 4) AS average_health_index
FROM blood_monitoring
GROUP BY route
ORDER BY average_health_index DESC
""").show()

+----------+--------------------+
|     route|average_health_index|
+----------+--------------------+
|Hospital_3|              0.0795|
|Hospital_4|              0.0794|
|Hospital_2|              0.0692|
|Hospital_1|              0.0685|
+----------+--------------------+



## 4. RDD Operations

Although Spark DataFrames are the preferred abstraction for structured data, RDDs (Resilient Distributed Datasets) provide fine-grained control over distributed processing. This section demonstrates basic RDD transformations and actions using the healthcare IoT dataset.

In [31]:
rdd = spark_df.rdd

print("Number of partitions:", rdd.getNumPartitions())

Number of partitions: 8


In [32]:
rdd.first()

Row(bag_id='BAG_0001', timestamp=datetime.datetime(2024, 1, 11, 0, 0), route='Hospital_3', blood_type='B-', product_type='RBC', temp_mean=3.5, temp_min=3.89771693765934, temp_max=4.023181695453228, temp_std=0.0985550258564581, frac_temp_above_6=0.0354990463646471, frac_temp_above_8=-0.002283350878834, hum_mean=58.09366967855133, hum_std=3.1408698792539544, door_count=-0.0146404552180646, light_mean_abs=-2.55828823001327, accel_rms=0.0475496697601288, handling_stress=0.442093104703897, health_index=0.9941757563161104)

In [33]:
rdd.count()

288000

In [34]:
routes = rdd.map(lambda row: row.route)

routes.take(10)

['Hospital_3',
 'Hospital_3',
 'Hospital_3',
 'Hospital_3',
 'Hospital_3',
 'Hospital_3',
 'Hospital_3',
 'Hospital_3',
 'Hospital_3',
 'Hospital_3']

In [35]:
hospital1 = rdd.filter(lambda row: row.route == "Hospital_1")

hospital1.take(5)

[Row(bag_id='BAG_0002', timestamp=datetime.datetime(2024, 1, 1, 0, 0), route='Hospital_1', blood_type='A-', product_type='RBC', temp_mean=3.9391356668307815, temp_min=3.7677281223529366, temp_max=4.529917247532183, temp_std=0.1855789802460102, frac_temp_above_6=0.0209537416602548, frac_temp_above_8=0.0031730035319638, hum_mean=54.67642283088831, hum_std=2.18767476187035, door_count=1.0236257096932366, light_mean_abs=8.932243869801239, accel_rms=0.0663522309336801, handling_stress=0.4630735938578226, health_index=0.9775118576273476),
 Row(bag_id='BAG_0002', timestamp=datetime.datetime(2024, 1, 1, 1, 0), route='Hospital_1', blood_type='A-', product_type='RBC', temp_mean=3.70750947226908, temp_min=3.367114607293204, temp_max=4.251731735619468, temp_std=0.151589877488321, frac_temp_above_6=0.0343560039061693, frac_temp_above_8=0.0001824936920535, hum_mean=54.13865237191409, hum_std=1.06135304735337, door_count=0.0574910942465911, light_mean_abs=0.2265475244538268, accel_rms=0.0188637149897

In [36]:
spark_df.groupBy("route").count().show()

+----------+-----+
|     route|count|
+----------+-----+
|Hospital_4|63360|
|Hospital_1|75840|
|Hospital_3|72000|
|Hospital_2|76800|
+----------+-----+



### Observation

The RDD API was used to demonstrate basic distributed processing operations such as `map()` and `filter()`. Although Spark DataFrames are generally preferred for structured data because they are optimized by the Catalyst Optimizer, RDDs provide a lower-level abstraction that helps understand how distributed transformations are executed across multiple partitions.

For aggregation, the DataFrame API (`groupBy().count()`) was used to demonstrate the MapReduce concept because it provides the same distributed execution while taking advantage of Spark's query optimization.

## 5. Partitioning Strategy

Partitioning is an important optimization technique in Apache Spark. It determines how data is distributed across different executors. A good partitioning strategy reduces data movement during shuffle operations and improves overall performance.

Since hospital route is frequently used for filtering and aggregation in this project, it is chosen as the partitioning key.

In [37]:
print("Current Partitions:", spark_df.rdd.getNumPartitions())

Current Partitions: 8


In [38]:
partitioned_df = spark_df.repartition(4, "route")

print("New Partitions:", partitioned_df.rdd.getNumPartitions())

New Partitions: 4


In [39]:
partitioned_df.show(5)

+--------+-------------------+----------+----------+------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+--------------------+------------------+------------------+
|  bag_id|          timestamp|     route|blood_type|product_type|         temp_mean|          temp_min|          temp_max|          temp_std| frac_temp_above_6| frac_temp_above_8|          hum_mean|           hum_std|        door_count|    light_mean_abs|           accel_rms|   handling_stress|      health_index|
+--------+-------------------+----------+----------+------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+--------------------+------------------+------------------+
|BAG_0002|2024-01-01 00:00:00|Hospital_1|        A-|   

In [40]:
spark_df.groupBy("route").count().orderBy(col("count").desc()).show()

+----------+-----+
|     route|count|
+----------+-----+
|Hospital_2|76800|
|Hospital_1|75840|
|Hospital_3|72000|
|Hospital_4|63360|
+----------+-----+



# 6. Data Skew

Data skew occurs when one partition contains significantly more records than the others. This creates an imbalance in workload because one executor takes much longer to process its partition while other executors remain idle.

The hospital route distribution in this dataset is relatively balanced. Therefore, an artificial skewed dataset is created to demonstrate the concept of data skew and how salting can be used to reduce it.

In [41]:
spark_df.groupBy("route") \
    .count() \
    .orderBy(col("count").desc()) \
    .show()

+----------+-----+
|     route|count|
+----------+-----+
|Hospital_2|76800|
|Hospital_1|75840|
|Hospital_3|72000|
|Hospital_4|63360|
+----------+-----+



### Creating an Artificially Skewed Dataset

Since the original healthcare dataset is relatively balanced, an artificial skewed dataset is created to demonstrate how data skew affects distributed processing and how salting can be used to improve load balancing.

In [43]:
from pyspark.sql.functions import lit

hospital1 = spark_df.filter(col("route") == "Hospital_1").limit(5000)
hospital2 = spark_df.filter(col("route") == "Hospital_2").limit(300)
hospital3 = spark_df.filter(col("route") == "Hospital_3").limit(200)
hospital4 = spark_df.filter(col("route") == "Hospital_4").limit(100)

skew_df = (
    hospital1
    .union(hospital2)
    .union(hospital3)
    .union(hospital4)
)

skew_df.groupBy("route").count().show()

+----------+-----+
|     route|count|
+----------+-----+
|Hospital_1| 5000|
|Hospital_2|  300|
|Hospital_3|  200|
|Hospital_4|  100|
+----------+-----+



# 7. Salting Technique

Salting is an optimization technique used to reduce the impact of data skew. Instead of processing all records with the same key on a single executor, a random value (salt) is added to the key. This distributes the workload across multiple partitions, resulting in better load balancing and improved performance.

In [44]:
from pyspark.sql.functions import floor, rand, concat_ws

In [45]:
salted_df = skew_df.withColumn(
    "salt",
    floor(rand() * 5)
)

In [46]:
salted_df = salted_df.withColumn(
    "salted_route",
    concat_ws("_", col("route"), col("salt"))
)

salted_df.select(
    "route",
    "salt",
    "salted_route"
).show(20, truncate=False)

+----------+----+------------+
|route     |salt|salted_route|
+----------+----+------------+
|Hospital_1|1   |Hospital_1_1|
|Hospital_1|0   |Hospital_1_0|
|Hospital_1|3   |Hospital_1_3|
|Hospital_1|3   |Hospital_1_3|
|Hospital_1|0   |Hospital_1_0|
|Hospital_1|1   |Hospital_1_1|
|Hospital_1|1   |Hospital_1_1|
|Hospital_1|1   |Hospital_1_1|
|Hospital_1|2   |Hospital_1_2|
|Hospital_1|4   |Hospital_1_4|
|Hospital_1|3   |Hospital_1_3|
|Hospital_1|2   |Hospital_1_2|
|Hospital_1|3   |Hospital_1_3|
|Hospital_1|1   |Hospital_1_1|
|Hospital_1|0   |Hospital_1_0|
|Hospital_1|1   |Hospital_1_1|
|Hospital_1|2   |Hospital_1_2|
|Hospital_1|4   |Hospital_1_4|
|Hospital_1|0   |Hospital_1_0|
|Hospital_1|0   |Hospital_1_0|
+----------+----+------------+
only showing top 20 rows


In [47]:
salted_df.groupBy("salted_route") \
    .count() \
    .orderBy(col("count").desc()) \
    .show(20, truncate=False)

+------------+-----+
|salted_route|count|
+------------+-----+
|Hospital_1_4|1051 |
|Hospital_1_2|1026 |
|Hospital_1_1|990  |
|Hospital_1_0|982  |
|Hospital_1_3|951  |
|Hospital_2_3|71   |
|Hospital_2_4|61   |
|Hospital_2_0|60   |
|Hospital_2_1|57   |
|Hospital_2_2|51   |
|Hospital_3_1|48   |
|Hospital_3_2|48   |
|Hospital_3_3|40   |
|Hospital_3_4|35   |
|Hospital_3_0|29   |
|Hospital_4_0|22   |
|Hospital_4_2|20   |
|Hospital_4_4|20   |
|Hospital_4_3|20   |
|Hospital_4_1|18   |
+------------+-----+



### Observation

Before applying salting, all 5,000 records belonging to Hospital_1 were associated with a single key. This would cause one executor to perform most of the work while the remaining executors stayed relatively idle.

After applying salting, the Hospital_1 records were distributed across multiple salted keys (Hospital_1_0, Hospital_1_1, etc.). This balances the workload across partitions and reduces the impact of data skew.

# 8. Lazy Evaluation

Apache Spark follows a lazy evaluation model, meaning that transformations are not executed immediately. Instead, Spark records all transformations and builds a Directed Acyclic Graph (DAG). The computations are only executed when an action such as `show()`, `count()`, or `collect()` is called.

This optimization allows Spark to reduce unnecessary computations and improve execution efficiency.

In [48]:
lazy_df = (
    spark_df
    .filter(col("temp_mean") > 5)
    .select(
        "bag_id",
        "route",
        "temp_mean",
        "health_index"
    )
)

print("Transformation created.")
print("No computation has been executed yet.")

Transformation created.
No computation has been executed yet.


In [49]:
lazy_df.show(10)

+--------+----------+-----------------+------------------+
|  bag_id|     route|        temp_mean|      health_index|
+--------+----------+-----------------+------------------+
|BAG_0001|Hospital_3|5.315062817907007|0.7849385026015636|
|BAG_0001|Hospital_3|5.319148932554025|0.3147064460151149|
|BAG_0001|Hospital_3|              5.5|0.2167358301029033|
|BAG_0001|Hospital_3|              5.5| 0.185206351805334|
|BAG_0001|Hospital_3|              5.5|0.1365016754371394|
|BAG_0001|Hospital_3|              5.5|0.0825720599878312|
|BAG_0001|Hospital_3|              5.5|0.0368438210458359|
|BAG_0001|Hospital_3|5.191318711516735|0.0139577154900092|
|BAG_0001|Hospital_3|5.443456661111791|0.0016301577963071|
|BAG_0001|Hospital_3|              5.5| 1.674243773587E-4|
+--------+----------+-----------------+------------------+
only showing top 10 rows


### Observation

The filtering and selection operations were not executed immediately when they were defined. Spark stored these transformations and waited until the `show()` action was called. This demonstrates Spark's lazy evaluation model, which improves performance by optimizing the execution plan before processing the data.

# 9. Directed Acyclic Graph (DAG)

Spark represents all transformations as a Directed Acyclic Graph (DAG). Before executing a job, Spark analyzes the DAG and optimizes the execution plan. Stage boundaries are introduced whenever wide transformations, such as `groupBy()` or `join()`, require data to be shuffled across partitions.

In [50]:
lazy_df.explain(True)

== Parsed Logical Plan ==
'Project ['bag_id, 'route, 'temp_mean, 'health_index]
+- Filter (temp_mean#22 > cast(5 as double))
   +- Relation [bag_id#17,timestamp#18,route#19,blood_type#20,product_type#21,temp_mean#22,temp_min#23,temp_max#24,temp_std#25,frac_temp_above_6#26,frac_temp_above_8#27,hum_mean#28,hum_std#29,door_count#30,light_mean_abs#31,accel_rms#32,handling_stress#33,health_index#34] csv

== Analyzed Logical Plan ==
bag_id: string, route: string, temp_mean: double, health_index: double
Project [bag_id#17, route#19, temp_mean#22, health_index#34]
+- Filter (temp_mean#22 > cast(5 as double))
   +- Relation [bag_id#17,timestamp#18,route#19,blood_type#20,product_type#21,temp_mean#22,temp_min#23,temp_max#24,temp_std#25,frac_temp_above_6#26,frac_temp_above_8#27,hum_mean#28,hum_std#29,door_count#30,light_mean_abs#31,accel_rms#32,handling_stress#33,health_index#34] csv

== Optimized Logical Plan ==
Project [bag_id#17, route#19, temp_mean#22, health_index#34]
+- Filter (isnotnull(tem

### Observation

The execution plan demonstrates how Spark builds and optimizes a Directed Acyclic Graph (DAG) before executing a job.

The Parsed Logical Plan represents the transformations exactly as written in the code. The Analyzed Logical Plan validates the schema and data types. The Optimized Logical Plan introduces additional optimizations such as automatically filtering out null values before applying the condition (`isnotnull(temp_mean)`).

The Physical Plan shows the actual execution strategy. Spark performs predicate pushdown by applying the filter (`temp_mean > 5`) while reading the CSV file, reducing the amount of data loaded into memory. This optimization improves execution efficiency and minimizes disk I/O.

# 10. Machine Learning using Spark MLlib

In this section, a machine learning model is developed using Spark MLlib to predict the Health Index of blood bags based on sensor readings collected during transportation.

The pipeline includes:
- Feature selection
- Handling missing target values
- Vector assembly
- Train-test split
- Model training
- Model evaluation

In [51]:
ml_df = spark_df.dropna(subset=["health_index"])

print("Rows after removing missing target:")
print(ml_df.count())

Rows after removing missing target:
288000


In [52]:
feature_columns = [
    "temp_mean",
    "temp_min",
    "temp_max",
    "temp_std",
    "frac_temp_above_6",
    "frac_temp_above_8",
    "hum_mean",
    "hum_std",
    "door_count",
    "light_mean_abs",
    "accel_rms",
    "handling_stress"
]

In [53]:
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(
    inputCols=feature_columns,
    outputCol="features"
)

ml_data = assembler.transform(ml_df)

ml_data.select("features", "health_index").show(5, truncate=False)

+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------------+
|features                                                                                                                                                                                                            |health_index      |
+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------------+
|[3.5,3.89771693765934,4.023181695453228,0.0985550258564581,0.0354990463646471,-0.002283350878834,58.09366967855133,3.1408698792539544,-0.0146404552180646,-2.55828823001327,0.0475496697601288,0.442093104703897]   |0.9941757563161104|
|[3.5,4.36967003926577,4.446650959843448,0.2126704154296398,0.05

In [54]:
train_data, test_data = ml_data.randomSplit(
    [0.8, 0.2],
    seed=42
)

print("Training Rows:", train_data.count())
print("Testing Rows:", test_data.count())

Training Rows: 230647
Testing Rows: 57353


In [55]:
from pyspark.ml.regression import LinearRegression

lr = LinearRegression(
    featuresCol="features",
    labelCol="health_index"
)

model = lr.fit(train_data)

26/08/01 22:51:04 WARN Instrumentation: [a05323c1] regParam is zero, which might cause numerical instability and overfitting.
26/08/01 22:51:05 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
26/08/01 22:51:05 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.lapack.JNILAPACK


In [56]:
predictions = model.transform(test_data)

predictions.select(
    "health_index",
    "prediction"
).show(10)

+------------------+-------------------+
|      health_index|         prediction|
+------------------+-------------------+
| 0.980271022375168| 0.2659672801109541|
| 0.936047805934561| 0.2857577573522013|
| 0.924114144397302|0.32922492769843453|
|0.8451936526920898|0.29027349221636667|
|  0.76051953462258|0.23617532816853534|
|0.7259242625894944| 0.2510060454821136|
| 0.695062889374023| 0.2531534073372477|
|0.6549209887973977| 0.2712952701709509|
|0.5624412760497294|0.25790925326251724|
|0.5579169442609024| 0.2646707567479266|
+------------------+-------------------+
only showing top 10 rows


In [57]:
from pyspark.ml.evaluation import RegressionEvaluator

rmse = RegressionEvaluator(
    labelCol="health_index",
    predictionCol="prediction",
    metricName="rmse"
)

r2 = RegressionEvaluator(
    labelCol="health_index",
    predictionCol="prediction",
    metricName="r2"
)

print("RMSE:", rmse.evaluate(predictions))
print("R² Score:", r2.evaluate(predictions))

RMSE: 0.13086126382045582
R² Score: 0.4569558062337409


A Linear Regression model was trained using Spark MLlib to predict the Health Index from sensor measurements. The model was evaluated using RMSE and R² metrics. The feature coefficients indicate the contribution of each sensor variable to the predicted Health Index.

The model successfully predicts the Health Index using healthcare sensor measurements, demonstrating Spark MLlib's capability for scalable distributed machine learning.

In [58]:
print("Intercept:", model.intercept)

print("\nCoefficients:")

for feature, coef in zip(feature_columns, model.coefficients):
    print(f"{feature}: {coef}")

Intercept: 0.449139948840966

Coefficients:
temp_mean: -0.004713257453771389
temp_min: 0.08253027282435076
temp_max: -0.06317173361892627
temp_std: 0.1552956919142949
frac_temp_above_6: -0.08983470757821335
frac_temp_above_8: -0.16829442286666038
hum_mean: -0.001887278332171124
hum_std: -0.04223418147010002
door_count: 0.0021222281169867587
light_mean_abs: 0.0001140194805919713
accel_rms: -0.1448176850928004
handling_stress: -0.018024100967593867


### Observation

A Linear Regression model was trained using Spark MLlib to predict the Health Index from healthcare sensor data. The model was evaluated using RMSE and R² metrics to measure its performance.

The results show how Spark MLlib can be used to build and evaluate machine learning models on large datasets while taking advantage of distributed processing.

# 11. Saving the Processed Data

In Big Data applications, processed data is commonly stored in efficient distributed formats for future analysis. Spark supports several storage formats, with Parquet being one of the most widely used due to its columnar storage, compression, and optimized query performance.

The cleaned dataset is saved as a Parquet file.

In [59]:
output_path = "output/processed_healthcare_data"

ml_df.write \
    .mode("overwrite") \
    .parquet(output_path)

print("Processed data saved successfully.")

26/08/01 22:52:45 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers


Processed data saved successfully.


In [60]:
saved_df = spark.read.parquet(output_path)

print("Rows:", saved_df.count())
print("Columns:", len(saved_df.columns))

saved_df.show(5)

Rows: 288000
Columns: 18
+--------+-------------------+----------+----------+------------+-----------------+------------------+-----------------+------------------+-------------------+------------------+------------------+-----------------+------------------+-------------------+------------------+------------------+------------------+
|  bag_id|          timestamp|     route|blood_type|product_type|        temp_mean|          temp_min|         temp_max|          temp_std|  frac_temp_above_6| frac_temp_above_8|          hum_mean|          hum_std|        door_count|     light_mean_abs|         accel_rms|   handling_stress|      health_index|
+--------+-------------------+----------+----------+------------+-----------------+------------------+-----------------+------------------+-------------------+------------------+------------------+-----------------+------------------+-------------------+------------------+------------------+------------------+
|BAG_0158|2024-01-27 13:00:00|Hospital_

### Observation

The cleaned dataset was successfully stored in Parquet format. Compared to CSV, Parquet provides better compression, faster query execution, and efficient column-wise storage, making it well suited for large-scale distributed data processing.

In [61]:
spark.stop()

print("Spark Session Stopped.")

Spark Session Stopped.


# Conclusion

This project demonstrated the complete workflow of distributed data processing using Apache Spark on a healthcare IoT dataset for Blood Cold Chain Monitoring. The project covered data loading, preprocessing, Spark SQL, RDD operations, partitioning, handling data skew through salting, lazy evaluation, DAG execution, and machine learning using Spark MLlib.

The implementation highlights Spark's ability to efficiently process large datasets while providing scalability, fault tolerance, and optimized execution through distributed computing techniques.